In [15]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [16]:
spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")
spark.sql("show tables").show()

+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|northwind_db|    employees|      false|
|northwind_db|order_details|      false|
|northwind_db|       orders|      false|
|            |     tmp_view|       true|
+------------+-------------+-----------+



In [17]:
df = spark.table("Orders")
df.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- RequiredDate: date (nullable = true)
 |-- ShippedDate: string (nullable = true)
 |-- ShipVia: string (nullable = true)
 |-- Freight: decimal(10,2) (nullable = true)
 |-- ShipName: string (nullable = true)
 |-- ShipAddress: string (nullable = true)
 |-- ShipCity: string (nullable = true)
 |-- ShipRegion: string (nullable = true)
 |-- ShipPostalCode: string (nullable = true)
 |-- ShipCountry: string (nullable = true)



## Structs
We can think of Structs as DataFrames within DataFrames

In [18]:
from pyspark.sql.functions import struct

df_structs = df.select(
    "OrderID",
    struct("ShippedDate", 
           "ShipName", 
           "ShipRegion", 
           "ShipCity").alias("ShipDetails")
)
display(df_structs.select("*").limit(2))
display(df_structs.select("OrderID", "ShipDetails.*").limit(2))
df_structs.createOrReplaceTempView("tmp_view")

DataFrame[OrderID: string, ShipDetails: struct<ShippedDate:string,ShipName:string,ShipRegion:string,ShipCity:string>]

DataFrame[OrderID: string, ShippedDate: string, ShipName: string, ShipRegion: string, ShipCity: string]

In [19]:
spark.sql("select OrderID, ShipDetails.* from tmp_view").show()

+-------+--------------------+------------------+-------------+--------+
|OrderID|         ShippedDate|          ShipName|   ShipRegion|ShipCity|
+-------+--------------------+------------------+-------------+--------+
|  10315|1996-10-03 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10318|1996-10-04 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10321|1996-10-11 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10473|1997-03-21 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10621|1997-08-11 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10674|1997-09-30 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10749|1997-12-19 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10798|1998-01-05 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10829|1998-01-23 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10933|1998-03-16 00:00:...|    Island Trading|Isle of Wight|   Cowes|
|  10255|1996-07-15 00:00:...|Richter Supermarkt|  

## Arrays and explode
explode takes a column that consits of arrays and creates one row per value in the array 

In [20]:
from pyspark.sql.functions import split, explode, collect_list

df_a = spark.createDataFrame([[1, "This is a simple string"],
                              [2, "Hello World!"],], 
                             ["id", "Text"])
df_a.printSchema()
display(df_a)
df_a = df_a.withColumn("SplittedText", split("Text", " "))
display(df_a)
df_a = df_a.withColumn("ExplodedText", explode("SplittedText"))
display(df_a)
df_a = df_a.groupBy("id", "Text").agg(collect_list("ExplodedText").alias("Collected text"))
display(df_a)



root
 |-- id: long (nullable = true)
 |-- Text: string (nullable = true)



DataFrame[id: bigint, Text: string]

DataFrame[id: bigint, Text: string, SplittedText: array<string>]

DataFrame[id: bigint, Text: string, SplittedText: array<string>, ExplodedText: string]

DataFrame[id: bigint, Text: string, Collected text: array<string>]

## Map

In [21]:
from pyspark.sql.functions import split, explode, count, create_map, col

df_a = spark.createDataFrame([[1, "This is a simple string simple"],
                              [2, "Hello World!"]], 
                             ["id", "Text"])

# Split text into words and explode
words = df_a.withColumn("Word", explode(split("Text", " ")))

# Calculate frequency of words
freq = words.groupBy("Word").agg(count("Word").alias("cnt"))
display(freq)

# Create a map column (Word -> cnt) and explode
freq_map = freq.withColumn("Mapped", create_map(col("Word"), col("cnt")))
display(freq_map)

# Explode the map into key-value columns
exploded = freq_map.selectExpr("explode(Mapped) as (Word, cnt)")
display(exploded)



DataFrame[Word: string, cnt: bigint]

DataFrame[Word: string, cnt: bigint, Mapped: map<string,bigint>]

DataFrame[Word: string, cnt: bigint]